# Piping 

In [1]:
%load_ext dotenv
%dotenv

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import CommaSeparatedListOutputParser

chat = ChatOpenAI(model="gpt-4",
                  seed=365,
                  max_completion_tokens=500,
                  temperature=0)

prompt_template = ChatPromptTemplate.from_template('''Tôi sắp nhận nôi một con {pet}, bạn hãy gợi ý cho con {pet} 3 cái tên thật hài hước.
''' + CommaSeparatedListOutputParser().get_format_instructions())

output_parser = CommaSeparatedListOutputParser()

pipe = prompt_template | chat | output_parser

pipe.invoke({"pet": "dog"})


['Bánh Mì', 'Cà Phê', 'Phở']

### Batch

In [3]:
prompt_template_for_batch = ChatPromptTemplate.from_messages(
    [("human", "Tôi sắp nhận nuôi một con {pet} thuộc giống loài {breed}. Hãy cho tôi một vài mẹo để dạy dỗ nó.")])

# prompt_template_for_batch

pipe_batch = prompt_template_for_batch | chat

# pipe_batch.invoke({"pet": "dog", "breed": "chó chăn cừu"})

pipe_batch.batch([
    {"pet": "chó", "breed": "chó chăn cừu"},
    {"pet": "mèo", "breed": "mèo rừng"}
])


[AIMessage(content='1. Bắt đầu sớm: Chó chăn cừu thông minh và học hỏi nhanh chóng, vì vậy bạn nên bắt đầu huấn luyện chúng từ khi còn nhỏ. \n\n2. Sử dụng phương pháp thưởng phạt: Khi chúng làm đúng, hãy thưởng cho chúng bằng cách cho ăn thức ăn nhỏ hoặc khen ngợi. Khi chúng làm sai, hãy chỉ ra lỗi và hướng dẫn chúng cách làm đúng.\n\n3. Luyện tập thường xuyên: Chó chăn cừu cần được luyện tập thường xuyên để giữ cho trí óc và cơ thể của chúng luôn hoạt động. Đi dạo hàng ngày, chơi trò chơi hoặc thậm chí là dạy chúng một số mánh mới có thể giúp giữ cho chúng luôn sảng khoái.\n\n4. Sử dụng lệnh đơn giản: Bắt đầu với các lệnh đơn giản như "ngồi", "đứng", "đợi", "đi" và "đến". Một khi chúng đã nắm vững các lệnh này, bạn có thể chuyển sang các lệnh phức tạp hơn.\n\n5. Kiên nhẫn: Huấn luyện chó chăn cừu (hoặc bất kỳ giống chó nào) đòi hỏi kiên nhẫn. Đừng nản lòng nếu chúng không học được ngay lập tức. Hãy nhớ rằng mỗi chó có tốc độ học hỏi khác nhau.\n\n6. Đừng quên', additional_kwargs={'ref

### Stream

In [4]:
response = pipe_batch.stream(
    {"pet": "chó", "breed": "chó chăn cừu"}
)


In [5]:
for chunk in response:
    print (chunk.content, end="", flush=True)

1. Bắt đầu sớm: Chó chăn cừu thông minh và học hỏi nhanh chóng, vì vậy bạn nên bắt đầu huấn luyện chúng từ khi còn nhỏ. 

2. Sử dụng lệnh đơn giản: Bắt đầu với các lệnh cơ bản như "ngồi", "đứng", "ở lại" và "đến đây". Hãy nhất quán trong việc sử dụng các lệnh này và thưởng cho chó mỗi khi nó thực hiện đúng.

3. Thưởng: Sử dụng thức ăn hoặc đồ chơi yêu thích của chó như phần thưởng khi chúng thực hiện đúng lệnh. Điều này sẽ khích lệ chúng học hỏi nhanh hơn.

4. Luyện tập hàng ngày: Chó chăn cừu cần nhiều hoạt động thể chất và trí tuệ. Hãy dành ít nhất 30 phút mỗi ngày để luyện tập và chơi với chúng.

5. Kiên nhẫn: Huấn luyện chó không phải lúc nào cũng dễ dàng. Bạn cần kiên nhẫn và nhất quán trong việc huấn luyện.

6. Đừng sử dụng hình phạt: Hình phạt không hiệu quả và có thể gây ra hậu quả tiêu cực. Thay vào đó, hãy tập trung vào việc thưởng cho hành vi tốt.

7. Tận dụng bản năng chăn cừu: Chó chăn cừu có bản năng chăn gia súc

In [6]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

chat_prompt_templates_tools = ChatPromptTemplate.from_template('''Liệt kê 5 tools quan trọng nhất của {job title}. 
Câu trả lời chỉ bao gồm danh sách các tools.
''')

# chat_prompt_templates_tools

str_output_parser = StrOutputParser()


In [7]:
tools_chain = chat_prompt_templates_tools | chat | str_output_parser
tools_chain.invoke({'job title': 'Data science'})

'1. Python\n2. R\n3. SQL\n4. Tableau\n5. Hadoop'

In [8]:
chat_prompt_templates_strategies = ChatPromptTemplate.from_template('''Với mỗi tool hãy cho tôi chiến lược để học nó một cách hiệu quả. 
Đây là danh sách các {tools}''')

strategies_chain = chat_prompt_templates_strategies | chat | str_output_parser

learning_tools_strategies = tools_chain | {'tools': RunnablePassthrough()} | strategies_chain

learning_tools_strategies.invoke({'job title': 'Data science'})

'1. Python: \n   - Bắt đầu với việc hiểu cú pháp cơ bản của Python và cách sử dụng các loại dữ liệu khác nhau như list, tuple, dictionary, etc.\n   - Học về vòng lặp, điều kiện và hàm trong Python.\n   - Tiếp theo, hãy tìm hiểu về OOP (Lập trình hướng đối tượng) trong Python.\n   - Học cách sử dụng các thư viện phổ biến như Numpy, Pandas, Matplotlib, etc.\n   - Thực hành bằng cách giải quyết các bài toán thực tế và tham gia các dự án nhỏ.\n\n2. R: \n   - Bắt đầu với cú pháp cơ bản của R và cách sử dụng các loại dữ liệu khác nhau.\n   - Học về vòng lặp, điều kiện và hàm trong R.\n   - Học cách sử dụng các gói phổ biến như dplyr, ggplot2, etc.\n   - Thực hành bằng cách giải quyết các bài toán thực tế và tham gia các dự án nhỏ.\n\n3. SQL: \n   - Bắt đầu với cú pháp cơ bản của SQL như SELECT, INSERT, UPDATE, DELETE.\n   - Học về các khái niệm như JOIN, UNION, GROUP BY, etc.\n   - Học cách viết các truy vấn phức tạp hơn bằng cách kết hợp các khái niệm đã học.\n   - Thực hành bằng cách tạo v

In [10]:
learning_tools_strategies.get_graph().print_ascii()

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
  +--------------------+   
  | ChatPromptTemplate |   
  +--------------------+   
            *              
            *              
            *              
      +------------+       
      | ChatOpenAI |       
      +------------+       
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  
            *              
            *              
            *              
     +-------------+       
     | Passthrough |       
     +-------------+       
            *              
            *              
            *       